# 06. Building porosity and flow-drag parameters

This notebook derives the parameters of the porosity and flow-drag layer of HEC-RAS from the building footprints. It computes the building fraction of every 5 m cell, groups the cells into five building-density classes and assigns each class a plan porosity, a frontal-area density and a quadratic drag coefficient. It produces the table of porosity and flow-drag parameters and the classification raster imported into RAS Mapper.

## Configuration

In [ ]:
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.windows import Window
from shapely.geometry import Polygon, MultiPolygon, box
from shapely.ops import unary_union

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path("..")
BUILDING_SHP_1 = ROOT / "data" / "buildings" / "building_footprints_part1.shp"
BUILDING_SHP_2 = ROOT / "data" / "buildings" / "building_footprints_part2.shp"
STUDY_AREA_SHP = ROOT / "data" / "boundaries" / "model_domain.shp"   # None: use the extent of the buildings
OUTPUT_FOLDER = ROOT / "results" / "06_porosity_drag"

TARGET_EPSG = 32645                 # WGS 1984 UTM Zone 45N
ANALYSIS_CELL_SIZE_M = 5.0          # cell size of the HEC-RAS computational mesh
SUBCELL_RESOLUTION_M = 1.0          # rasterisation resolution inside each 5 m cell
TILE_ANALYSIS_CELLS = 200           # tile size in 5 m cells (200 cells = 1 km)

# building-density classes by building fraction of a 5 m cell
CLASS_BINS = [0.00, 0.05, 0.25, 0.50, 0.75, 1.000001]
CLASS_NAMES = ["OPEN", "LOW", "MEDIUM", "DENSE", "VERY_DENSE"]

DRAG_COEFFICIENT = 1.2              # building drag coefficient C_D (Macdonald et al. 1998)
THICKNESS_M = 6.0                   # height over which porosity and drag act

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

## 1. Geometry helpers

In [ ]:
def polygon_only(geom):
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, (Polygon, MultiPolygon)):
        return geom
    if hasattr(geom, "geoms"):
        polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]
        if polys:
            return unary_union(polys)
    return None

def clean_layer(gdf, target_epsg=32645):
    gdf = gdf.copy()

    if gdf.crs is None:
        raise ValueError(
            "A shapefile has no CRS. Define its CRS correctly before continuing."
        )

    if gdf.crs.to_epsg() != target_epsg:
        print(f"Reprojecting {gdf.crs} to EPSG:{target_epsg}")
        gdf = gdf.to_crs(epsg=target_epsg)

    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()

    invalid = ~gdf.geometry.is_valid
    if invalid.any():
        print(f"Repairing {invalid.sum():,} invalid geometries")
        gdf.loc[invalid, "geometry"] = gdf.loc[invalid, "geometry"].buffer(0)

    gdf["geometry"] = gdf.geometry.apply(polygon_only)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()

    return gdf.reset_index(drop=True)

## 2. Building footprints

In [ ]:
p1 = Path(BUILDING_SHP_1)
p2 = Path(BUILDING_SHP_2)

if not p1.exists():
    raise FileNotFoundError(f"Not found: {p1}")

if not p2.exists():
    raise FileNotFoundError(f"Not found: {p2}")

b1 = gpd.read_file(p1)
b2 = gpd.read_file(p2)

print("Layer 1")
print("Features:", len(b1))
print("CRS:", b1.crs)
print("Bounds:", b1.total_bounds)
print()

print("Layer 2")
print("Features:", len(b2))
print("CRS:", b2.crs)
print("Bounds:", b2.total_bounds)

## 3. Cleaning and merging

Invalid geometries are repaired, the two layers are merged and overlapping polygons are dissolved, so that no building is counted twice.

In [ ]:
b1 = clean_layer(b1, TARGET_EPSG)
b2 = clean_layer(b2, TARGET_EPSG)

raw = gpd.GeoDataFrame(
    pd.concat(
        [
            b1[["geometry"]].assign(SOURCE="SHP1"),
            b2[["geometry"]].assign(SOURCE="SHP2"),
        ],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=f"EPSG:{TARGET_EPSG}",
)

raw_area = raw.geometry.area.sum()

union_geom = unary_union(raw.geometry.values)

buildings = gpd.GeoDataFrame(
    {"PFD_CLASS": ["BUILDING"], "geometry": [union_geom]},
    crs=f"EPSG:{TARGET_EPSG}",
).explode(index_parts=False).reset_index(drop=True)

buildings = buildings[~buildings.geometry.is_empty].copy()
buildings["BLDG_ID"] = np.arange(1, len(buildings) + 1)
buildings["AREA_M2"] = buildings.geometry.area
buildings["PERIM_M"] = buildings.geometry.length

unique_area = buildings["AREA_M2"].sum()

print(f"Original feature count: {len(raw):,}")
print(f"Connected nonoverlapping building components: {len(buildings):,}")
print(f"Raw polygon area: {raw_area:,.2f} m²")
print(f"Unique footprint area: {unique_area:,.2f} m²")
print(f"Overlap removed: {raw_area - unique_area:,.2f} m²")
print(f"CRS: {buildings.crs}")

## 4. Processing extent

The footprints are clipped to the model domain, and the extent of the domain is snapped outward to the 5 m analysis grid.

In [ ]:
if STUDY_AREA_SHP is not None:
    study = gpd.read_file(STUDY_AREA_SHP)
    study = clean_layer(study, TARGET_EPSG)
    study_geom = unary_union(study.geometry.values)
    buildings = buildings[buildings.intersects(study_geom)].copy()
    buildings["geometry"] = buildings.geometry.intersection(study_geom)
    buildings = buildings[~buildings.geometry.is_empty].copy()
    bounds = study.total_bounds
    print("Extent: model-domain polygon.")
else:
    bounds = buildings.total_bounds
    print("Extent: combined building footprints.")

minx, miny, maxx, maxy = bounds

minx_a = math.floor(minx / ANALYSIS_CELL_SIZE_M) * ANALYSIS_CELL_SIZE_M
miny_a = math.floor(miny / ANALYSIS_CELL_SIZE_M) * ANALYSIS_CELL_SIZE_M
maxx_a = math.ceil(maxx / ANALYSIS_CELL_SIZE_M) * ANALYSIS_CELL_SIZE_M
maxy_a = math.ceil(maxy / ANALYSIS_CELL_SIZE_M) * ANALYSIS_CELL_SIZE_M

width_m = maxx_a - minx_a
height_m = maxy_a - miny_a

coarse_w = int(round(width_m / ANALYSIS_CELL_SIZE_M))
coarse_h = int(round(height_m / ANALYSIS_CELL_SIZE_M))

print(f"Extent width: {width_m / 1000:.2f} km")
print(f"Extent height: {height_m / 1000:.2f} km")
print(f"5 m raster dimensions: {coarse_w:,} columns by {coarse_h:,} rows")
print(f"Total 5 m cells: {coarse_w * coarse_h:,}")

if width_m > 100_000 or height_m > 100_000:
    print()
    print("WARNING: Extent exceeds 100 km in at least one direction.")
    print("This may indicate an outlier or incorrect geometry.")

footprint_area = buildings.geometry.area.sum()
print(f"Footprint area inside the domain: {footprint_area:,.2f} m2 "
      f"(all footprints: {unique_area:,.2f} m2)")

## 5. Building fraction, porosity and density class

The footprints are rasterised at 1 m, in tiles of 1 km by 1 km so that only one tile is held in memory, and averaged to the 5 m grid. This gives the building fraction $f_b$ of every 5 m cell. The plan porosity is $\phi = 1 - f_b$, and each cell is assigned to one of five classes of building fraction: open (0 to 0.05), low (0.05 to 0.25), medium (0.25 to 0.50), dense (0.50 to 0.75) and very dense (0.75 to 1.00).

In [ ]:
ratio_float = ANALYSIS_CELL_SIZE_M / SUBCELL_RESOLUTION_M
ratio = int(round(ratio_float))

if not np.isclose(ratio_float, ratio):
    raise ValueError(
        "ANALYSIS_CELL_SIZE_M must be an integer multiple of SUBCELL_RESOLUTION_M."
    )

coarse_transform = from_origin(
    minx_a,
    maxy_a,
    ANALYSIS_CELL_SIZE_M,
    ANALYSIS_CELL_SIZE_M
)

fraction_tif = OUTPUT_FOLDER / "building_fraction_5m.tif"
porosity_tif = OUTPUT_FOLDER / "porosity_5m.tif"
class_tif = OUTPUT_FOLDER / "building_density_class_5m.tif"

profile_float = {
    "driver": "GTiff",
    "height": coarse_h,
    "width": coarse_w,
    "count": 1,
    "dtype": "float32",
    "crs": f"EPSG:{TARGET_EPSG}",
    "transform": coarse_transform,
    "nodata": -9999.0,
    "compress": "lzw",
    "tiled": True,
}

profile_class = profile_float.copy()
profile_class.update(dtype="uint8", nodata=0)

sindex = buildings.sindex

class_counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
class_fraction_sums = np.zeros(len(CLASS_NAMES), dtype=np.float64)
class_porosity_sums = np.zeros(len(CLASS_NAMES), dtype=np.float64)

total_tiles_y = math.ceil(coarse_h / TILE_ANALYSIS_CELLS)
total_tiles_x = math.ceil(coarse_w / TILE_ANALYSIS_CELLS)
total_tiles = total_tiles_y * total_tiles_x

print(f"Tiles to process: {total_tiles:,}")
print(
    f"Nominal tile size: "
    f"{TILE_ANALYSIS_CELLS * ANALYSIS_CELL_SIZE_M:.0f} m by "
    f"{TILE_ANALYSIS_CELLS * ANALYSIS_CELL_SIZE_M:.0f} m"
)

tile_number = 0

with rasterio.open(fraction_tif, "w", **profile_float) as dst_frac, \
     rasterio.open(porosity_tif, "w", **profile_float) as dst_phi, \
     rasterio.open(class_tif, "w", **profile_class) as dst_cls:

    for row0 in range(0, coarse_h, TILE_ANALYSIS_CELLS):
        nrows = min(TILE_ANALYSIS_CELLS, coarse_h - row0)

        for col0 in range(0, coarse_w, TILE_ANALYSIS_CELLS):
            ncols = min(TILE_ANALYSIS_CELLS, coarse_w - col0)
            tile_number += 1

            x_left = minx_a + col0 * ANALYSIS_CELL_SIZE_M
            y_top = maxy_a - row0 * ANALYSIS_CELL_SIZE_M
            x_right = x_left + ncols * ANALYSIS_CELL_SIZE_M
            y_bottom = y_top - nrows * ANALYSIS_CELL_SIZE_M

            tile_box = box(x_left, y_bottom, x_right, y_top)

            idx = list(sindex.query(tile_box, predicate="intersects"))
            subset = buildings.iloc[idx] if idx else buildings.iloc[0:0]

            fine_h = nrows * ratio
            fine_w = ncols * ratio

            fine_transform = from_origin(
                x_left,
                y_top,
                SUBCELL_RESOLUTION_M,
                SUBCELL_RESOLUTION_M
            )

            if len(subset) == 0:
                frac = np.zeros((nrows, ncols), dtype=np.float32)
            else:
                mask_fine = rasterize(
                    [(geom, 1) for geom in subset.geometry if not geom.is_empty],
                    out_shape=(fine_h, fine_w),
                    transform=fine_transform,
                    fill=0,
                    all_touched=False,
                    dtype="uint8",
                )

                frac = mask_fine.reshape(
                    nrows, ratio, ncols, ratio
                ).mean(axis=(1, 3)).astype("float32")

                del mask_fine

            phi = (1.0 - frac).astype("float32")

            class_arr = (
                np.digitize(frac, CLASS_BINS[1:-1], right=False) + 1
            ).astype("uint8")

            window = Window(col0, row0, ncols, nrows)

            dst_frac.write(frac, 1, window=window)
            dst_phi.write(phi, 1, window=window)
            dst_cls.write(class_arr, 1, window=window)

            for class_id in range(1, len(CLASS_NAMES) + 1):
                m = class_arr == class_id
                c = int(m.sum())
                if c:
                    class_counts[class_id - 1] += c
                    class_fraction_sums[class_id - 1] += float(frac[m].sum())
                    class_porosity_sums[class_id - 1] += float(phi[m].sum())

            if tile_number == 1 or tile_number % 25 == 0 or tile_number == total_tiles:
                print(f"Processed tile {tile_number:,} of {total_tiles:,}")

print()
print("Finished tiled raster processing.")
print("Building fraction:", fraction_tif)
print("Porosity:", porosity_tif)
print("Density class:", class_tif)

## 6. Class means

Each built class is assigned the mean porosity of its cells. The open class is treated as unobstructed ($\phi = 1$).

In [ ]:
rows = []

for i, name in enumerate(CLASS_NAMES):
    count = int(class_counts[i])

    if count > 0:
        mean_fraction = class_fraction_sums[i] / count
        mean_porosity = class_porosity_sums[i] / count
    else:
        mean_fraction = np.nan
        mean_porosity = np.nan

    if name == "OPEN":
        mean_porosity = 1.0

    rows.append(
        {
            "CLASS_ID": i + 1,
            "PFD_CLASS": name,
            "CELL_COUNT": count,
            "MEAN_BUILDING_FRACTION": mean_fraction,
            "CALCULATED_POROSITY": mean_porosity,
        }
    )

class_table = pd.DataFrame(rows)
display(class_table)

## 7. Frontal-area density and quadratic drag coefficient

For buildings smaller than a cell the linear drag coefficient is zero and the quadratic drag coefficient is

$$b = C_D\,a_F$$

where $C_D$ is the building drag coefficient and $a_F$ the frontal area of buildings facing the flow per unit volume of water (Guinot et al. 2017; HEC 2026). For a building that is not overtopped the depth cancels, and $a_F$ reduces to the projected building width per unit plan area (Nepf 1999). Because the flow direction is not known in advance, the projected width is replaced by the orientation-averaged width $\overline{W} = P_{ch}/\pi$, where $P_{ch}$ is the perimeter of the convex hull of the footprint (Cauchy's formula). The frontal-area density of class $c$ is then

$$a_{F,c} = \overline{f}_{b,c}\,\frac{\sum_k \overline{W}_k}{\sum_k A_k}$$

where $\overline{f}_{b,c}$ is the mean building fraction of the class, and $\overline{W}_k$ and $A_k$ are the orientation-averaged width and the footprint area of building $k$, summed over all buildings of the domain. This scaling assumes that building size and shape do not differ systematically between classes. No flood marks were available for calibration, so $C_D$ is fixed at 1.2 (Macdonald et al. 1998) and the thickness at 6 m.

In [ ]:
convex = buildings.geometry.convex_hull
buildings["CONV_PER_M"] = convex.length
buildings["MEAN_FRONT_W_M"] = buildings["CONV_PER_M"] / math.pi

width_area_ratio = buildings["MEAN_FRONT_W_M"].sum() / footprint_area
print(f"Total orientation-averaged width / total footprint area = {width_area_ratio:.4f} 1/m")

built = class_table["PFD_CLASS"] != "OPEN"

class_table["FRONTAL_AREA_DENSITY_1_PER_M"] = 0.0
class_table.loc[built, "FRONTAL_AREA_DENSITY_1_PER_M"] = (
    class_table.loc[built, "MEAN_BUILDING_FRACTION"] * width_area_ratio
)

class_table["THICKNESS_M"] = np.where(built, THICKNESS_M, 0.0)
class_table["LINEAR_DRAG_1_PER_S"] = 0.0
class_table["QUADRATIC_DRAG_1_PER_M"] = DRAG_COEFFICIENT * class_table["FRONTAL_AREA_DENSITY_1_PER_M"]
class_table["DRAG_COEFFICIENT"] = DRAG_COEFFICIENT

class_table.to_csv(OUTPUT_FOLDER / "porosity_drag_class_parameters.csv", index=False)
class_table[["CLASS_ID", "PFD_CLASS", "CELL_COUNT", "MEAN_BUILDING_FRACTION", "CALCULATED_POROSITY",
             "FRONTAL_AREA_DENSITY_1_PER_M", "QUADRATIC_DRAG_1_PER_M", "LINEAR_DRAG_1_PER_S", "THICKNESS_M"]]